# DeepGuard — Prepare DF40 on Google Drive

This notebook prepares a persistent DF40 workspace, checks free Drive space, and safely stages an officially obtained DF40 archive. It deliberately does not guess or scrape an unverified dataset URL.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
from pathlib import Path
import shutil, json, datetime
ROOT=Path('/content/drive/MyDrive/DeepGuard')
DF40=ROOT/'datasets'/'DF40'
DF40.mkdir(parents=True,exist_ok=True)
usage=shutil.disk_usage('/content/drive')
print(f'Free Drive space: {usage.free/1024**3:.1f} GB')
print('DF40 target:',DF40)
if usage.free < 110*1024**3: raise RuntimeError('Reserve at least 110 GB before staging the DF40 test set.')


In [ ]:
# Inspect whether DF40 is already present.
from pathlib import Path
files=list(DF40.rglob('*'))
print('Entries:',len(files))
for p in files[:30]: print(p.relative_to(DF40))
if not files: print('DF40 is not installed yet.')


## Official data hand-off

DF40 is large and subject to its upstream distribution terms. Download/obtain the dataset from the official DF40 source, then upload or copy the archive into Drive (for example `DeepGuard/datasets/df40_official/`). The next cell will verify the archive and record its hash.


In [ ]:
# Set this to the path of the officially obtained archive on Drive.
ARCHIVE=None  # e.g. '/content/drive/MyDrive/DeepGuard/datasets/df40_official/DF40_test.tar'
EXPECTED_SHA256=None  # optional, if supplied by the official distributor

import hashlib, shutil, json, datetime
def sha256(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''): h.update(b)
    return h.hexdigest()

if ARCHIVE:
    src=Path(ARCHIVE)
    if not src.exists(): raise FileNotFoundError(src)
    digest=sha256(src)
    print('SHA-256:',digest)
    if EXPECTED_SHA256 and digest.lower()!=EXPECTED_SHA256.lower(): raise RuntimeError('SHA-256 mismatch')
    print('Archive verified. Extract using the official DF40 instructions.')
else:
    print('No archive selected yet — this is expected until DF40 is obtained officially.')


In [ ]:
manifest={'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'df40_target':str(DF40),'free_space_gb':shutil.disk_usage('/content/drive').free/1024**3,'archive':ARCHIVE,'sha256':None if not ARCHIVE else digest}
log=ROOT/'logs'/'df40_download_manifest.json'; log.parent.mkdir(parents=True,exist_ok=True); log.write_text(json.dumps(manifest,indent=2)); print(log)
